In [1]:
!pip install faiss-cpu==1.7.4
!pip install boto3 pandas tqdm

In [2]:
import boto3
import json
import pandas as pd
from tqdm import tqdm
import faiss
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from textwrap import dedent

In [3]:
import time

def measure_time(func, *args, **kwargs):
    """Measures the execution time of a function."""
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    return result, end_time - start_time

In [4]:
# Initialize S3 client
s3 = boto3.client("s3")

In [5]:
# Define bucket and file info
bucket_name = "ai-arxiv2-dataset"
file_key = "dataset/arxiv-metadata-oai-snapshot.json" 

In [6]:
obj = s3.get_object(Bucket=bucket_name, Key=file_key)
total_lines = sum(1 for _ in obj["Body"].iter_lines())  # count total lines once

In [7]:
# Re-fetch file for actual reading
obj = s3.get_object(Bucket=bucket_name, Key=file_key)

# Read 50% of lines
target_lines = int(total_lines * 0.25)

titles = []
abstracts = []

print(f"Reading ~25% of the dataset ({target_lines} lines) from S3...")

Reading ~25% of the dataset (718191 lines) from S3...


In [8]:
for i, line in enumerate(obj["Body"].iter_lines()):
    if i >= target_lines:
        break
    if line:
        record = json.loads(line)
        title = record.get("title", "")
        abstract = record.get("abstract", "")
        titles.append(title)
        abstracts.append(abstract)
    if i % 10000 == 0 and i > 0:
        print(f"Processed {i} lines...")

print("Completed reading 25% of the dataset!")

Processed 10000 lines...
Processed 20000 lines...
Processed 30000 lines...
Processed 40000 lines...
Processed 50000 lines...
Processed 60000 lines...
Processed 70000 lines...
Processed 80000 lines...
Processed 90000 lines...
Processed 100000 lines...
Processed 110000 lines...
Processed 120000 lines...
Processed 130000 lines...
Processed 140000 lines...
Processed 150000 lines...
Processed 160000 lines...
Processed 170000 lines...
Processed 180000 lines...
Processed 190000 lines...
Processed 200000 lines...
Processed 210000 lines...
Processed 220000 lines...
Processed 230000 lines...
Processed 240000 lines...
Processed 250000 lines...
Processed 260000 lines...
Processed 270000 lines...
Processed 280000 lines...
Processed 290000 lines...
Processed 300000 lines...
Processed 310000 lines...
Processed 320000 lines...
Processed 330000 lines...
Processed 340000 lines...
Processed 350000 lines...
Processed 360000 lines...
Processed 370000 lines...
Processed 380000 lines...
Processed 390000 line

In [9]:
# Step 2 — Create DataFrame
df = pd.DataFrame({
    "title": titles,
    "abstract": abstracts
})

print(f"\nTotal Records Loaded: {len(df)}")
print(df.head())


Total Records Loaded: 718191
                                               title  \
0  Calculation of prompt diphoton production cros...   
1           Sparsity-certifying Graph Decompositions   
2  The evolution of the Earth-Moon system based o...   
3  A determinant of Stirling cycle numbers counts...   
4  From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...   

                                            abstract  
0    A fully differential calculation in perturba...  
1    We describe a new algorithm, the $(k,\ell)$-...  
2    The evolution of Earth-Moon system is descri...  
3    We show that a determinant of Stirling cycle...  
4    In this paper we show how to compute the $\L...  


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718191 entries, 0 to 718190
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   title     718191 non-null  object
 1   abstract  718191 non-null  object
dtypes: object(2)
memory usage: 11.0+ MB


In [11]:
# Step 4 — Save a small subset locally
df.sample(1000).to_csv("sample_ai_arxiv_25.csv", index=False)
print("Saved local sample as sample_ai_arxiv_25.csv")

Saved local sample as sample_ai_arxiv_25.csv


In [12]:
## embedding

In [13]:
# Bedrock client (for embeddings)
bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1")

In [14]:
# Titan embedding model
model_id = "amazon.titan-embed-text-v1"

In [15]:
df = pd.read_csv("sample_ai_arxiv_25.csv")

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     1000 non-null   object
 1   abstract  1000 non-null   object
dtypes: object(2)
memory usage: 15.8+ KB


In [17]:
# Function to get embeddings
def get_embedding(text):
    payload = json.dumps({"inputText": text})
    response = bedrock.invoke_model(modelId=model_id, body=payload)
    result = json.loads(response["body"].read())
    return result["embedding"]

In [18]:
# Generate embeddings for abstracts
embeddings = []
for abstract in tqdm(df["abstract"], desc="Generating embeddings"):
    if abstract.strip():
        emb = get_embedding(abstract)
        embeddings.append(emb)
    else:
        embeddings.append([0.0] * 1536)  # placeholder vector length

df["embedding"] = embeddings

Generating embeddings: 100%|██████████| 1000/1000 [04:00<00:00,  4.16it/s]


In [19]:
## METRIC 1: Ingestion and FAISS Creation Time

def generate_embeddings_and_update_df(dataframe, embedding_func):
    embeddings_list = []
    for abstract in tqdm(dataframe["abstract"], desc="Measuring Embedding Time"):
        if abstract.strip():
            # Use the existing get_embedding function
            emb = embedding_func(abstract)
            embeddings_list.append(emb)
        else:
            # Placeholder for empty abstracts (1536 is the expected Titan dimension)
            embeddings_list.append([0.0] * 1536) 
    dataframe["embedding"] = embeddings_list
    return dataframe

# Execute and measure embedding time
df_with_emb, embedding_time = measure_time(generate_embeddings_and_update_df, df.copy(), get_embedding)

# Step 2: Measure FAISS Index Creation Time
def create_faiss_index(df_with_embeddings):
    emb_matrix = np.array(df_with_embeddings["embedding"].to_list()).astype("float32")
    # Create FAISS index
    index = faiss.IndexFlatL2(emb_matrix.shape[1])
    index.add(emb_matrix)
    return index, emb_matrix.shape[1] # Return index and dimension

faiss_index, embedding_dim = create_faiss_index(df_with_emb)

# Step 3: Measure FAISS Index Save and Upload Time
def save_and_upload_index(index, bucket_name, key):
    faiss.write_index(index, "temp_faiss_index.bin")
    s3.upload_file("temp_faiss_index.bin", bucket_name, key)

_, faiss_upload_time = measure_time(save_and_upload_index, 
                                    faiss_index, 
                                    "ai-arxiv2-dataset", 
                                    "metrics/faiss_index_metric.bin")

print(f"\n Ingestion & Index Creation Metrics")
#print(f"Embedding Model: {model_id} (Dimension: {embedding_dim})")
print(f"Total Records Indexed: {len(df_with_emb)}")
print(f"Embedding Generation Time (3000 records): {embedding_time:.2f} seconds")
print(f"FAISS Index Creation Time (3000 records): (Included in below, often < 0.1s)") 
print(f"FAISS Index Save and Upload Time (on-cloud): {faiss_upload_time:.2f} seconds")

ingestion_faiss_total_time = embedding_time + faiss_upload_time
print(f"Total Time: {ingestion_faiss_total_time:.2f} seconds")

Measuring Embedding Time: 100%|██████████| 1000/1000 [05:00<00:00,  3.33it/s]



 Ingestion & Index Creation Metrics
Total Records Indexed: 1000
Embedding Generation Time (3000 records): 300.49 seconds
FAISS Index Creation Time (3000 records): (Included in below, often < 0.1s)
FAISS Index Save and Upload Time (on-cloud): 0.15 seconds
Total Ingestion & Indexing Time: 300.63 seconds


In [20]:
df.to_parquet("arxiv_25_with_embeddings.parquet")

In [21]:
# Upload back to S3
s3 = boto3.client("s3")
s3.upload_file("arxiv_25_with_embeddings.parquet", "ai-arxiv2-dataset", "processed/arxiv_25_with_embeddings.parquet")

print(" Embeddings generated and uploaded to S3: processed/arxiv_25_with_embeddings.parquet")

 Embeddings generated and uploaded to S3: processed/arxiv_25_with_embeddings.parquet


In [22]:
## Storing embeddings for semantic search

In [23]:
df = pd.read_parquet("arxiv_25_with_embeddings.parquet")

# Convert embeddings to numpy
emb_matrix = np.array(df["embedding"].to_list()).astype("float32")

In [24]:
# Create FAISS index
index = faiss.IndexFlatL2(emb_matrix.shape[1])
index.add(emb_matrix)

faiss.write_index(index, "faiss_index.bin")
print(" FAISS index created and saved.")

# Upload index to S3
s3.upload_file("faiss_index.bin", "ai-arxiv2-dataset", "processed/faiss_index.bin")

 FAISS index created and saved.


In [25]:
## --- METRIC 2: Data Retrieval Time (FAISS) ---

# Re-use the existing `get_similar` function but wrap it for timing
def get_similar_timed(query, index, df_obj, embedding_func):
    # 1. Get query embedding (Timed)
    query_emb, query_emb_time = measure_time(embedding_func, query)
    query_vector = np.array(query_emb).astype("float32").reshape(1, -1)
    
    # 2. Search FAISS index (Timed)
    search_start_time = time.time()
    D, I = index.search(query_vector, 3) 
    search_end_time = time.time()
    faiss_search_time = search_end_time - search_start_time
    
    return df_obj.iloc[I[0]][["title", "abstract"]], query_emb_time, faiss_search_time

query_report = "Recent advancements in computer vision"
_, query_emb_time, faiss_search_time = get_similar_timed(query_report, faiss_index, df_with_emb, get_embedding)

retrieval_total_time = query_emb_time + faiss_search_time

print(f"\nData Retrieval Metrics (Top 3 for '{query_report}')")
print(f"Query Embedding Time: {query_emb_time:.4f} seconds")
print(f"FAISS Search Time: {faiss_search_time:.4f} seconds")


Data Retrieval Metrics (Top 3 for 'Recent advancements in computer vision')
Query Embedding Time: 0.0895 seconds
FAISS Search Time: 0.0006 seconds
Total Retrieval Time: 0.0901 seconds
